# Maximise LLM prompt-cache hits

Reorder the columns of a table so that serialized rows share a longer prefix and
more of every prompt is served from the LLM's cache. A model rewrites a published
ordering algorithm; a fixed grader measures it on five real tables. Meta-Evolve
gives the model six rewrites and keeps the best score.

These are the same cells as the [llm-sql docs page](./), which also explains the
recorded run. Use a Python 3.12+ kernel and set `OPENROUTER_API_KEY` for the model
calls; scoring the seed needs no key. The saved outputs are from a live run with
`z-ai/glm-5.3-flash`.

## Set up

Install the dependencies.

In [ ]:
%pip install -q "pandas>=2.1" networkx "meta-evolve @ git+https://github.com/sentient-xyz/meta-evolve.git"

## 1. Fetch the seed algorithm

The seed is GRG, the published algorithm from the ADRS `llm_sql` task. This
cell downloads it and loads it as a Python source string — the artifact
Meta-Evolve rewrites.

In [ ]:
import subprocess
from pathlib import Path

import meta_evolve as meta

WORK = Path("adrs_llm_sql_run").resolve()
TASK = WORK / "adrs-upstream" / "openevolve" / "examples" / "ADRS" / "llm_sql"
WORK.mkdir(parents=True, exist_ok=True)

if not TASK.exists():
    subprocess.run(
        ["git", "clone", "--filter=blob:none", "--no-checkout",
         "https://github.com/UCB-ADRS/ADRS.git", str(WORK / "adrs-upstream")],
        check=True,
    )
    subprocess.run(
        ["git", "sparse-checkout", "set", "openevolve/examples/ADRS/llm_sql"],
        cwd=WORK / "adrs-upstream", check=True,
    )
    subprocess.run(["git", "checkout"], cwd=WORK / "adrs-upstream", check=True)

SEED = (TASK / "initial_program.py").read_text()
print(f"seed: {len(SEED)} chars")
# Output:
# seed: 16703 chars

seed: 16703 chars

## 2. Score a candidate

The grader reorders five tables with the candidate and returns one number:

```text
combined_score = 0.95 * mean(hit_rate) + 0.05 * (12 - min(12, mean(runtime))) / 12
```

Hit rate carries most of the weight; runtime is a tiebreaker. Run this cell
before setting an API key: a hit rate of about `0.713` means the setup works.
It downloads `score_one.py` if that file is not next to the notebook.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path
from urllib.request import urlretrieve

GRADER = Path("score_one.py")
if not GRADER.exists():
    urlretrieve(
        "https://sentient-xyz.github.io/meta-evolve-docs/applications/llm-sql/score_one.py",
        GRADER,
    )

def evaluate(source):
    """Score one Evolved module. Returns combined_score, or raises on failure."""
    work = WORK / "candidates"
    work.mkdir(parents=True, exist_ok=True)
    program = work / "candidate.py"
    program.write_text(str(source))
    done = subprocess.run(
        [sys.executable, str(GRADER.resolve()), str(program)],
        cwd=str(WORK),
        env={**os.environ, "ADRS_TASK_DIR": str(TASK)},
        capture_output=True, text=True, timeout=900,
    )
    report = None
    for line in reversed(done.stdout.splitlines()):
        if line.strip().startswith("{"):
            report = json.loads(line.strip())
            break
    if not report or not report.get("ok"):
        raise RuntimeError((report or {}).get("error") or done.stderr[-500:] or "score failed")
    print(f"hit rate {report['average_hit_rate']:.4f} · "
          f"runtime {report['average_runtime']:.1f}s · "
          f"combined {report['combined_score']:.4f}")
    return report["combined_score"]

evaluate(SEED)
# Output:
# hit rate 0.7129 · runtime 7.5s · combined 0.6961

hit rate 0.7129 · runtime 7.5s · combined 0.6961

## 3. Propose a rewrite

The proposer is an ordinary function: source in, source out. It sends the
current best program to `z-ai/glm-5.3-flash` through OpenRouter and returns the
rewritten module.

In [ ]:
import json, os, re, urllib.request

API_KEY = os.environ["OPENROUTER_API_KEY"]
MODEL = "z-ai/glm-5.3-flash"
SYSTEM = """You evolve a Python algorithm for the ADRS llm_sql task.
Reorder columns and rows only. Never change a cell.
Emit one module that defines class Evolved(Algorithm) with:
    def reorder(self, df, early_stop=0, row_stop=None, col_stop=None,
                col_merge=[], one_way_dep=[], distinct_value_threshold=0.8,
                parallel=True)
reorder must return a (DataFrame, list) tuple.
Import the base class with `from solver import Algorithm`.
Return only a ```python block."""

def propose(parent):
    payload = json.dumps({
        "model": MODEL,
        "temperature": 0,
        "seed": 0,
        "messages": [
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": "Improve this Evolved algorithm.\n\n```python\n" + str(parent) + "\n```"},
        ],
    }).encode()
    request = urllib.request.Request(
        "https://openrouter.ai/api/v1/chat/completions",
        data=payload,
        headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=900) as response:
        body = json.load(response)
    text = body["choices"][0]["message"]["content"]
    blocks = re.findall(r"```(?:python)?\s*\n(.*?)```", text, re.DOTALL)
    source = max(blocks, key=len).strip() if blocks else text
    if "class Evolved" not in source:
        raise RuntimeError("model did not return class Evolved")
    return source

## 4. Try six rewrites

`meta.improve` scores the seed, then runs six rewrites with Greedy search. A
higher score becomes the next parent; a lower score keeps the earlier best.
Expect a long run and real model usage.

In [ ]:
result = meta.improve(
    seed=SEED,
    proposer=propose,
    evaluator=evaluate,
    trials=6,
    random_seed=0,
)

hit rate 0.7129 · runtime 7.5s · combined 0.6961
hit rate 0.6925 · runtime 0.2s · combined 0.7070
hit rate 0.7029 · runtime 0.3s · combined 0.7166
hit rate 0.7029 · runtime 0.3s · combined 0.7166
hit rate 0.7030 · runtime 0.2s · combined 0.7169
hit rate 0.7030 · runtime 0.2s · combined 0.7168
hit rate 0.7032 · runtime 0.2s · combined 0.7171

## See the results

`result.trials()` is the seed plus every rewrite. `result.best().value` is the
selected source.

In [ ]:
for number, attempt in enumerate(result.trials()):
    score = attempt.metrics.get("score")
    label = f"{score:.4f}" if isinstance(score, (int, float)) else str(attempt.state)
    print(f"trial {number}: {label}")

print(f"\nselected: {len(str(result.best().value))} chars")
# Output:
# trial 0: 0.6961
# trial 1: 0.7070
# trial 2: 0.7166
# trial 3: 0.7166
# trial 4: 0.7169
# trial 5: 0.7168
# trial 6: 0.7171
#
# selected: 26113 chars

trial 0: 0.6961
trial 1: 0.7070
trial 2: 0.7166
trial 3: 0.7166
trial 4: 0.7169
trial 5: 0.7168
trial 6: 0.7171

selected: 26113 chars

## How to read this

The score rose from 0.6961 to 0.7171, but the hit rate fell from 0.7129 to 0.7032.
The selected program is 34 times faster than the seed, and under this grader that
speed was worth more than the cache hits it gave up. The docs page breaks the
score into its two parts: [The recorded run](./#the-recorded-run).

To search for cache hits alone, or to try another search method, a different model,
feedback to the model, or an agent harness, see
[Try your own table](./#try-your-own-table) on the docs page.

Task: [UCB-ADRS/ADRS](https://github.com/UCB-ADRS/ADRS) ·
[arXiv 2510.06189](https://arxiv.org/abs/2510.06189)